In [ ]:
import numpy as np
# STEP 1: Activation functions
def tanh(x):
    return np.tanh(x)

def tanh_derivative(x):
    return 1 - x**2
# STEP 2: Create Simple RNN class
class SimpleRNN:

    def __init__(self, input_size, hidden_size, output_size, lr=0.01):

        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.lr = lr

        # Initialize weights using Xavier initialization
        self.W_xh = np.random.randn(input_size, hidden_size) * np.sqrt(2/(input_size + hidden_size))
        self.W_hh = np.random.randn(hidden_size, hidden_size) * np.sqrt(2/(2 * hidden_size))
        self.W_ho = np.random.randn(hidden_size, output_size) * np.sqrt(2/(hidden_size + output_size))

        # Initialize biases
        self.b_h = np.zeros((1, hidden_size))
        self.b_o = np.zeros((1, output_size))


    # STEP 3: Forward propagation
    def forward(self, inputs):

        self.inputs = inputs
        self.hidden = {}
        self.outputs = {}

        h_prev = np.zeros((1, self.hidden_size))
        self.hidden[-1] = h_prev

        for t, x in enumerate(inputs):

            x = x.reshape(1, -1)

            h = tanh(
                np.dot(x, self.W_xh) +
                np.dot(h_prev, self.W_hh) +
                self.b_h
            )

            self.hidden[t] = h
            h_prev = h

            y = np.dot(h, self.W_ho) + self.b_o
            self.outputs[t] = y

        return self.outputs


    # STEP 4: Backpropagation Through Time
    def backward(self, targets):

        dW_xh = np.zeros_like(self.W_xh)
        dW_hh = np.zeros_like(self.W_hh)
        dW_ho = np.zeros_like(self.W_ho)

        db_h = np.zeros_like(self.b_h)
        db_o = np.zeros_like(self.b_o)

        dh_next = np.zeros((1, self.hidden_size))
        T = len(self.inputs)

        for t in reversed(range(T)):

            x = self.inputs[t].reshape(1, -1)
            h = self.hidden[t]
            h_prev = self.hidden[t-1]

            # Calculate output error
            dy = (self.outputs[t] - targets[t].reshape(1, -1)) / T

            dW_ho += np.dot(h.T, dy)
            db_o += dy

            # Backpropagate error
            dh = np.dot(dy, self.W_ho.T) + dh_next
            dh_raw = dh * (1 - h**2)

            dW_xh += np.dot(x.T, dh_raw)
            dW_hh += np.dot(h_prev.T, dh_raw)
            db_h += dh_raw

            dh_next = np.dot(dh_raw, self.W_hh.T)

        # Gradient clipping
        for grad in [dW_xh, dW_hh, dW_ho, db_h, db_o]:
            np.clip(grad, -5, 5, out=grad)

        # Update weights using Gradient Descent
        self.W_xh -= self.lr * dW_xh
        self.W_hh -= self.lr * dW_hh
        self.W_ho -= self.lr * dW_ho

        self.b_h -= self.lr * db_h
        self.b_o -= self.lr * db_o
# STEP 5: Create training data
input_seq = np.array([
    [0.1], [0.2], [0.3], [0.4], [0.5]
])

target_seq = np.array([
    [0.2], [0.4], [0.6], [0.8], [1.0]
])

# STEP 6: Initialize RNN
rnn = SimpleRNN(
    input_size=1,
    hidden_size=5,
    output_size=1,
    lr=0.05
)

# STEP 7: Train RNN
epochs = 1000
loss_history = []

for epoch in range(epochs):

    outputs = rnn.forward(input_seq)

    loss = np.mean([
        0.5 * (outputs[t] - target_seq[t])**2
        for t in range(len(input_seq))
    ])

    loss_history.append(loss)

    rnn.backward(target_seq)

    if (epoch + 1) % 200 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss:.6f}")

# STEP 8: Test the trained RNN
test_seq = np.array([
    [0.6], [0.7], [0.8], [0.9], [1.0]
])

predictions = rnn.forward(test_seq)

print("\nTesting RNN:")
print("Input Sequence:", test_seq.flatten())
print("Predicted Output:", predictions[len(test_seq)-1].flatten())

# STEP 9: Plot training loss
import matplotlib.pyplot as plt

plt.plot(loss_history)
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("RNN Training Loss")
plt.show()
